In [1]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import detectors as det
import filters as filt
import integrators as integ
from collections import defaultdict

# csv_file_name = 'closed_loop_18-08-2026_21-07-25_4_CCW_18m.csv'
# csv_path = '../data/measured_closed_loops/'
# csv_save_path = '../data/ZARU_and_loop_closure/'
# df = pd.read_csv(f'{csv_path}{csv_file_name}', skipinitialspace=True)

csv_file_name = 'mag_walk_indoors_23-08-2026_20-02-47.csv'
csv_path = '../data/magnetometer_readings/'
csv_save_path = '../data/magnetometer_readings/'
df = pd.read_csv(f'{csv_path}{csv_file_name}', skipinitialspace=True)

In [77]:
# ===========================================================================================
# ZARU WAS DEEMED REDUNDENT FOR THIS PROJECT - YIELDS WORSE RESULTS THAN MAHONY GZ CORRECTION.
# ===========================================================================================

# loop_logs = [
#     'closed_loop_18-08-2026_21-18-50_1_CW_NoStop_18m.csv',
#     'closed_loop_18-08-2026_21-19-41_2_CW_NoStop_18m.csv',
#     'closed_loop_18-08-2026_21-20-55_3_CW_NoStop_18m.csv',
# ]

# loop_logs = [
#     'closed_loop_18-08-2026_20-23-47_1_CW_18m.csv',
#     'closed_loop_18-08-2026_20-42-32_2_CW_18m.csv',
#     'closed_loop_18-08-2026_20-44-45_3_CW_18m.csv',
#     'closed_loop_18-08-2026_20-51-07_4_CW_18m.csv',
# ]

# loop_logs = [
#     'closed_loop_19-08-2026_21-17-41_1_CCW_16m_2min_walk_NO_Stops.csv',
#     'closed_loop_19-08-2026_21-17-41_1_CCW_16m_2min_walk_with_stops.csv',
# ]

# loop_logs = [
#     'closed_loop_18-08-2026_21-15-02_long_loop_CCW.csv',
# ]

# csv_path = '../data/measured_closed_loops/'
# ZARU_THRESHOLD = 2.0
# ZARU_DWELL = 100

# for file in loop_logs:
#     df = pd.read_csv(f'{csv_path}{file}', skipinitialspace=True)
#     zvw_mask = det.detect_zvw(df)

#     gx, gy, gz = df['gx'].values, df['gy'].values, df['gz'].values
#     ax, ay, az = df['ax'].values, df['ay'].values, df['az'].values
#     omega_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    
#     time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6
#     print(f'\nProcessing: {file}')
#     print(f'Total time: {time_sec.iloc[-1]:.2f} (s)')

#     quiet_samples = (omega_mag < ZARU_THRESHOLD) & zvw_mask
#     blocks = (quiet_samples != pd.Series(quiet_samples).shift()).cumsum()
#     zaru_mask = ((pd.Series(quiet_samples).groupby(blocks).transform('sum') >= ZARU_DWELL) & quiet_samples).values

#     print(f'Amount of ZARU windows detected: {len(np.unique(blocks[zaru_mask]))}')
#     gz_cleaned, final_bias_z = filt.apply_zaru_single_mean(gz, zaru_mask)

#     true_bias = np.mean(gz[:700])
#     print(f'True bias (based on the first initial pause: {true_bias:.3f} °/s)')

#     gz_use = gz_cleaned
#     if zaru_mask.sum() > 0:
#         assert not np.array_equal(gz_use, df['gz'].values), "ZARU active but raw gz reached the filter"

#     dt_array = np.diff(df['t_us'] - df['t_us'].iloc[0])
#     dt_array = np.insert(dt_array, 0, dt_array.mean()) / 1e6

#     masked_quats = filt.mahony_filter(ax, ay, az, 
#                                       gx, gy, gz_use, 
#                                       dt_array, zvw_mask, Kp=2.0, Ki=0.5)

#     raw_accel = np.column_stack((ax, ay, az))
#     global_accel = filt.rotate_vector_by_quaternion(raw_accel, masked_quats)
#     linear_accel = np.copy(global_accel)
#     linear_accel[:, 2] -= 1.0 
#     accel_ms2 = linear_accel * 9.80665 

#     vel_zupt, pos_zupt, _ = integ.integrate_kinematics(accel_ms2, dt_array, zvw_mask)

#     start_pos = pos_zupt[0, :2]
#     end_pos = pos_zupt[-1, :2]
#     closure_gap = np.linalg.norm(end_pos - start_pos)

#     print(f"  -> ZARU Bias Estimate: {final_bias_z:.4f} °/s")
#     print(f"  -> Closure Gap: {closure_gap:.3f} m")


In [8]:
loop_logs = [
# 'closed_loop_18-08-2026_20-23-47_1_CW_18m.csv',
# 'closed_loop_18-08-2026_20-42-32_2_CW_18m.csv',
# 'closed_loop_18-08-2026_20-44-45_3_CW_18m.csv',
# 'closed_loop_18-08-2026_20-51-07_4_CW_18m.csv',
# 'closed_loop_18-08-2026_21-00-25_1_CCW_18m.csv',
# 'closed_loop_18-08-2026_21-04-11_2_CCW_18m.csv',
# 'closed_loop_18-08-2026_21-05-53_3_CCW_18m.csv',
# 'closed_loop_18-08-2026_21-07-25_4_CCW_18m.csv',
# 'closed_loop_18-08-2026_21-18-50_1_CW_NoStop_18m.csv',
# 'closed_loop_18-08-2026_21-19-41_2_CW_NoStop_18m.csv',
# 'closed_loop_18-08-2026_21-20-55_3_CW_NoStop_18m.csv',
# 'closed_loop_18-08-2026_21-22-12_1_CCW_NoStop_18m.csv',
# 'closed_loop_18-08-2026_21-26-21_2_CCW_NoStop_18m.csv',
# 'closed_loop_18-08-2026_21-26-59_3_CCW_NoStop_18m.csv',
]

zvw_mask = det.detect_zvw(df)

gx, gy, gz = df['gx'].values, df['gy'].values, df['gz'].values
ax, ay, az = df['ax'].values, df['ay'].values, df['az'].values

time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6

dt_array = np.diff(df['t_us'] - df['t_us'].iloc[0])
dt_array = np.insert(dt_array, 0, dt_array.mean()) / 1e6

masked_quats = filt.mahony_filter(ax, ay, az, 
                                    gx, gy, gz, 
                                    dt_array, zvw_mask, Kp=2.0, Ki=0.5)

raw_accel = np.column_stack((ax, ay, az))
global_accel = filt.rotate_vector_by_quaternion(raw_accel, masked_quats)
linear_accel = np.copy(global_accel)
linear_accel[:, 2] -= 1.0 
accel_ms2 = linear_accel * 9.80665 

vel_zupt, pos_zupt, _ = integ.integrate_kinematics(accel_ms2, dt_array, zvw_mask)

start_pos = pos_zupt[0, :2]
end_pos = pos_zupt[-1, :2]
closure_gap = np.linalg.norm(end_pos - start_pos)


# Manhattan snapping:

# Find ZVW mid points
zvw_blocks = (zvw_mask != np.roll(zvw_mask, 1)).cumsum()
zvw_midpoints_idx = []
for block_id in np.unique(zvw_blocks[zvw_mask]):
    block_indices = np.where(zvw_blocks == block_id)[0]
    zvw_midpoints_idx.append(block_indices[len(block_indices)//2])

# Extract X, Y coordinates at the midpoints:
step_coords = pos_zupt[zvw_midpoints_idx, :2]

# Calculate step vectors:
delta_coords = np.diff(step_coords, axis=0) # X2 - X1, Y2 - Y1
step_distances = np.linalg.norm(delta_coords, axis=1)

# Filter out tiny micro movements:
valid_steps = step_distances > 0.2 # > 20cm
valid_delta_coords = delta_coords[valid_steps]
valid_step_distances = np.linalg.norm(valid_delta_coords, axis=1)

# Convert to 2D coordinates on a Cartesian coordinate system
step_heading_rads = np.arctan2(valid_delta_coords[:, 1], valid_delta_coords[:,0])

# Find dominant grid using a circle mean (multiply by 4 for four corners coordinate system)
theta_4 = step_heading_rads * 4

# Magnitude * sin/cos(angle) - calculate weighted geometric mean so lower strides have lower geometric power
mean_cos = np.sum(valid_step_distances * np.cos(theta_4)) # X comp
mean_sin = np.sum(valid_step_distances * np.sin(theta_4)) # Y comp

# Divide by 4 to get the building's global offset, reversing the multiplication by 4 from earlier
grid_offset_rad = np.arctan2(mean_sin, mean_cos) / 4.0 # Circle mean
grid_offset_deg = np.rad2deg(grid_offset_rad)
print(f'Calculated grid offset: {grid_offset_deg:.2f} deg')

# Apply grid/manhattan snapping
all_headings_rad = np.arctan2(delta_coords[:,1], delta_coords[:,0])
all_headings_deg = np.rad2deg(all_headings_rad)

# Remove offset and divide -> round to snap -> return initial offset.
snapped_headings_deg = np.round((all_headings_deg - grid_offset_deg) / 90) * 90 + grid_offset_deg
snapped_headings_rad = np.deg2rad(snapped_headings_deg)

# Calculate the new straight X/Y step vectors using the snapped headings
new_delta_x = step_distances * np.cos(snapped_headings_rad)
new_delta_y = step_distances * np.sin(snapped_headings_rad)

snapped_coords = np.zeros_like(step_coords)
snapped_coords[0] = step_coords[0]

# Construct the map
for i in range(len(new_delta_x)):
    snapped_coords[i+1, 0] = snapped_coords[i, 0] + new_delta_x[i]
    snapped_coords[i+1, 1] = snapped_coords[i, 1] + new_delta_y[i]

snapped_closure_gap = np.linalg.norm(snapped_coords[-1] - snapped_coords[0])
print(f"Snapped Closure Gap: {snapped_closure_gap:.3f} m")
print(f"Pre-snapped Closure Gap: {closure_gap:.3f} m")


# Calculate distances between corners
corner_distances = []
current_leg_distance = step_distances[0]
current_heading = snapped_headings_deg[0]

# Calculate steps per wall
wall_counter = 1
steps_per_wall = defaultdict(list)

if step_distances[0] > 0.2:
    steps_per_wall[1].append(step_distances[0])

for i in range(1, len(step_distances)): # start from 1 to not accumulate step_distances[0] twice
    # force 360 to 0
    raw_diff = snapped_headings_deg[i] - current_heading
    heading_diff = (raw_diff + 180) % 360 - 180
    
    # Only turn if its a real stride (> 0.2)
    is_real_step = step_distances[i] > 0.2

    # We changed a corner
    if abs(heading_diff) > 1.0 and is_real_step:
        corner_distances.append(current_leg_distance)
        wall_counter += 1

        # reset
        current_leg_distance = step_distances[i]
        current_heading = snapped_headings_deg[i]
    else:
        # accumulate all non corner steps
        current_leg_distance += step_distances[i]

    if is_real_step:
        steps_per_wall[wall_counter].append(step_distances[i])


# add final step
corner_distances.append(current_leg_distance)
corner_distances = np.array(corner_distances)

for idx, length in enumerate(corner_distances):
    print(f"Wall {idx+1}: {length:.2f} m")

print(f"Total calculated distance: {sum(corner_distances):.2f} m")

print('- - - - - - -')
for wall_num, steps in steps_per_wall.items():
    print(f'Wall: {wall_num}, Steps: {len(steps)}, Calculated last step: {steps[-1]:.2f} m -> (total: {sum(steps):.2f})')

Calculated grid offset: 9.18 deg
Snapped Closure Gap: 0.342 m
Pre-snapped Closure Gap: 0.569 m
Wall 1: 2.62 m
Wall 2: 6.70 m
Wall 3: 2.54 m
Wall 4: 7.04 m
Total calculated distance: 18.89 m
- - - - - - -
Wall: 1, Steps: 2, Calculated last step: 1.23 m -> (total: 2.62)
Wall: 2, Steps: 5, Calculated last step: 1.36 m -> (total: 6.70)
Wall: 3, Steps: 2, Calculated last step: 1.25 m -> (total: 2.54)
Wall: 4, Steps: 5, Calculated last step: 1.42 m -> (total: 7.04)


In [7]:
# Draw map
fig, axs = plt.subplots(figsize=(10,6))
# Draw unsnapped
axs.plot(pos_zupt[:, 0], pos_zupt[:, 1], label='Raw unsnapped', color='tab:red')
axs.plot(snapped_coords[:,0], snapped_coords[:,1], label='Manhattan Snapped', color='tab:blue', marker='o')

axs.scatter([snapped_coords[0,0]], [snapped_coords[0,1]], label='Start', color='green', s=200, marker='*', zorder=5)
axs.scatter([snapped_coords[-1,0]], [snapped_coords[-1,1]], label='End', color='purple', s=150, marker='X', zorder=5)

plt.axis('equal')
plt.grid(True, linestyle='--', alpha=0.6)
plt.title('Manhattan Snapping: Corner-Vertex Placement Correction')
plt.legend()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_Mapping.png'), dpi=120)
plt.show()

In [24]:
# magnetometer analysis:
mag_readings_mask = df['valid_mag_reading'] == 1
mag_readings = df.loc[mag_readings_mask, ['mx', 'my', 'mz']]
mag_readings_magnitude = np.linalg.norm(mag_readings, axis=1)

pos_mag_readings = pos_zupt[mag_readings_mask]

fig2, axs2 = plt.subplots(figsize=(10,6))
sc = axs2.scatter(pos_mag_readings[:, 0], 
                  pos_mag_readings[:, 1], 
                  c=mag_readings_magnitude, 
                  cmap='viridis')

axs2.scatter([pos_mag_readings[0,0]], [pos_mag_readings[0,1]], label='Start', color='green', s=200, marker='*', zorder=5)
axs2.scatter([pos_mag_readings[-1,0]], [pos_mag_readings[-1,1]], label='Finish', color='Purple', s=150, marker='X', zorder=5)

cbar = fig2.colorbar(sc, ax=axs2)
cbar.set_label(r'Magnetometer Magnitude $\sqrt{m_x^2 + m_y^2 + m_z^2}$')

axs2.set_xlabel('X Position')
axs2.set_ylabel('Y Position')
axs2.set_title('Position Colored by Magnetometer Magnitude')
axs2.legend()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_Magnetometer_color_map.png'), dpi=120)
plt.show()
